### **Environment Setup, Package Installation & GitHub Cloning**

In [ ]:
%%capture
!pip install -U ipython -q
!pip install ultralytics grad-cam -q

from kaggle_secrets import UserSecretsClient 
gh_token = UserSecretsClient().get_secret("GH_TOKEN")
!git clone https://{gh_token}@github.com/Hannysia/drone_detection.git
!pip install -e drone_detection/
%cd /kaggle/working/drone_detection

%load_ext autoreload
%autoreload 2

### **Global Configurations & Dataset Symlinking**

In [ ]:
import os
import time
from pathlib import Path
import torch
import numpy as np
import cv2
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

from src.config import REPO_ID, HF_TOKEN, EXTRACTED_DATA_DIR, WORKING_DIR, OUTPUT_DIR
from src.processing.dataset import prepare_yolo_dataset
from src.processing.quantization import run_export_and_benchmark_pipeline

N_WARMUP = 10
N_BENCHMARK = 100
IMGSZ = 1280
MODEL_FILE = "models/yolo26n_final_model_best.pt"

DATASET_YAML = prepare_yolo_dataset(EXTRACTED_DATA_DIR, WORKING_DIR, run_name="quantization", use_negatives=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### **Model Weights Download**

In [ ]:
print("\nDownloading weights from HuggingFace...")
weights_path = Path(hf_hub_download(
    repo_id=REPO_ID,
    filename=MODEL_FILE,
    token=HF_TOKEN,
    repo_type="dataset",
    local_dir="/kaggle/working/hf_weights"
))
print(f"  Saved to: {weights_path}")


[DATA PREPARATION] Setting up dataset directories for run: quantization
[TRAIN] Symlinked 8,177 positives and 810 negatives.
[VAL] Symlinked 1,542 positives and 147 negatives.
[TEST] Symlinked 2,281 positives and 243 negatives.
[DATA PREPARATION] Created YAML configuration at: /kaggle/working/dataset_quantization.yaml


### **TensorRT FP16 Export & Empirical Benchmarking**

In [ ]:
run_export_and_benchmark_pipeline(
    weights_path=weights_path,
    dataset_yaml=DATASET_YAML,
    model_file_name=MODEL_FILE,
    imgsz=IMGSZ,
    n_warmup=N_WARMUP,
    n_benchmark=N_BENCHMARK
)


  Saved to: /kaggle/working/hf_weights/models/yolo26n_final_model_best.pt

1. FP32 PyTorch (baseline)
  Size: 5.2 MB

Benchmark: FP32 PyTorch
  FPS:         55.8
  Latency:     17.93 ± 0.55 ms

Validating accuracy for: FP32 PyTorch...
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 370.0±70.4 MB/s, size: 203.8 KB)
val: Scanning /kaggle/working/yolo_dataset_quantization/labels/test.cache... 2281 images, 243 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2524/2524 481.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 158/158 3.1it/s 50.5s0.3ss
                   all       2524       2281      0.918      0.808      0.888      0.689
Speed: 3.6ms preprocess, 13.5ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-3
  [ACCURACY] mAP@50 for FP32 PyTorch: 0.8875

2. TensorRT FP16 Export
  Export

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 18 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.6s, saved as '/kaggle/working/hf_weights/models/yolo26n_final_model_best.onnx' (9.8 MB)

TensorRT: starting export with TensorRT 11.1.0.106...
[06/20/2026-22:43:40] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
TensorRT: converting ONNX to FP16 mixed precision with ModelOpt AutoCast...
2026-06-20 22:43:40,644 - [modelopt][onnx] - WARNING - Shared constants were detected and duplicated accordingly.
2026-06-20 22:43:40,775 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']
2026-06-20 22:43:40,776 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...
2026-06-20 22:43:41,775 - [modelopt][onnx] - INFO - Skipping node /m

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ✅ Successfully saved to HF!


### **Benchmark Quantized Engine (Resized 1280x1280)**

In [ ]:
print("Downloading your quantized engine from HF...")
engine_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="models/yolo26n_final_model_best.engine",
    token=HF_TOKEN,
    repo_type="dataset",
    local_dir="/kaggle/working/hf_weights"
)

model = YOLO(engine_path, task="detect")
print("Model loaded successfully.")

test_image_path = list(Path("/kaggle/working/yolo_dataset_quantization/images/test").glob("*.jpg"))[0]
real_img = cv2.imread(str(test_image_path))
real_img_rgb = cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB)
real_img_resized = cv2.resize(real_img_rgb, (1280, 1280))

print(f"Running benchmark on real image: {test_image_path.name}")

for _ in range(10):
    model(real_img_resized, verbose=False, device="cuda")
torch.cuda.synchronize()

latencies = []
for _ in range(100):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    model(real_img_resized, verbose=False, device="cuda")
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - t0) * 1000)

avg_lat = np.mean(latencies)
print(f"\n{'='*50}")
print(f"RESULTS FOR QUANTIZED ENGINE (1280x1280)")
print(f"{'='*50}")
print(f"Average Latency: {avg_lat:.2f} ms")
print(f"Estimated FPS:   {1000/avg_lat:.1f}")
print(f"{'='*50}")

Model loaded successfully.
Running benchmark on real image: delta_thermal_field_126_f0022.jpg
Loading /kaggle/working/hf_weights/models/yolo26n_final_model_best.engine for TensorRT inference...
[06/21/2026-12:57:26] [TRT] [I] Loaded engine size: 6 MiB
[06/21/2026-12:57:26] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +52, now: CPU 0, GPU 60 (MiB)

RESULTS FOR QUANTIZED ENGINE (1280x1280)
Average Latency: 10.19 ms
Estimated FPS:   98.2


### **Benchmark Quantized Engine (Raw Image Resolution)**

In [ ]:
model = YOLO(engine_path, task="detect")
print("Model loaded successfully.")

test_image_path = list(Path("/kaggle/working/yolo_dataset_quantization/images/test").glob("*.jpg"))[0]
real_img = cv2.imread(str(test_image_path))

print(f"Running benchmark on real image: {test_image_path.name}")

for _ in range(10):
    model(real_img, verbose=False, device="cuda")
torch.cuda.synchronize()

latencies = []
for _ in range(100):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    model(real_img, verbose=False, device="cuda")
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - t0) * 1000)

avg_lat = np.mean(latencies)
print(f"\n{'='*50}")
print(f"RESULTS FOR QUANTIZED ENGINE (1280x1280)")
print(f"{'='*50}")
print(f"Average Latency: {avg_lat:.2f} ms")
print(f"Estimated FPS:   {1000/avg_lat:.1f}")
print(f"{'='*50}")

Model loaded successfully.
Running benchmark on real image: delta_thermal_field_126_f0022.jpg
Loading /kaggle/working/hf_weights/models/yolo26n_final_model_best.engine for TensorRT inference...
[06/21/2026-12:56:00] [TRT] [I] Loaded engine size: 6 MiB
[06/21/2026-12:56:00] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +52, now: CPU 0, GPU 60 (MiB)

RESULTS FOR QUANTIZED ENGINE (1280x1280)
Average Latency: 12.57 ms
Estimated FPS:   79.6
